# AKPS progression from physically scaled nucleus morphology

Only intrinsic single-nucleus features and nucleus-level predictions are used. There is no organoid aggregation. Organoid IDs serve only grouped CV and equal-count subsampling.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
DATA=ROOT/'data'/'recomputed_pure_nucleus_features'
OUT=ROOT/'results'/'classification'; OUT.mkdir(parents=True,exist_ok=True)
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr
from analysis.features import PURE_NUCLEUS_FEATURES
from analysis.akps_progression import LINE_ORDER,load_manual,load_auto,load_akps,run_all,stratified_subsample_by_organoid
from analysis.akps_stage_classifier import prep_nucleus_level,cv_evaluate,fit_full_and_select
PURE_NUCLEUS_FEATURES

## Scaling audit
Exact TIFF spacing is retained per row. The 2025-07-22 and 2025-08-04 NCO TIFFs lack calibration metadata and use an explicit 0.300 µm isotropic fallback. They remain marked as imputed and are excluded in a sensitivity analysis below.

In [ ]:
akps_raw=pd.read_csv(DATA/'trial_005_akps.csv')
spacing_audit=akps_raw.groupby(['line','spacing_z_um','spacing_y_um','spacing_x_um','spacing_source','spacing_is_imputed'],dropna=False).agg(n_nuclei=('label','size'),n_images=('img_name','nunique'),median_volume_um3=('volume_um3','median'),median_major_axis_um=('ellipsoid_axis_major_um','median')).reset_index()
spacing_audit.to_csv(OUT/'akps_spacing_audit.csv',index=False); spacing_audit

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,feature in zip(axes,['volume_um3','ellipsoid_axis_major_um','sphericity']):
    values=[akps_raw.loc[akps_raw.line==line,feature].clip(upper=akps_raw.loc[akps_raw.line==line,feature].quantile(.99)) for line in LINE_ORDER]
    ax.boxplot(values,tick_labels=LINE_ORDER,showfliers=False); ax.set_title(feature)
fig.suptitle('Scaled per-nucleus morphology across AKPS'); fig.tight_layout()

## P021N/P013T → AKPS transfer
Manual masks, Trial 005 automated masks, and their union are separate training sources. Each AKPS nucleus receives its own tumor probability. Equal-count subsampling prevents large organoids from dominating without aggregating rows.

In [ ]:
manual=load_manual(DATA/'manual_p021n_p013t.csv'); auto=load_auto(DATA/'trial_005_p021n_p013t.csv')
train_sets={'manual':manual,'auto':auto,'manual+auto':pd.concat([manual,auto],ignore_index=True)}
akps=load_akps(DATA/'trial_005_akps.csv')
scores,summary=run_all(train_sets,akps,modes=('nucleus',))
scores.to_csv(OUT/'notebook_akps_nucleus_scores.csv',index=False); summary.to_csv(OUT/'notebook_akps_nucleus_transfer_summary.csv',index=False); summary

In [ ]:
primary=scores[scores.model=='l0l2_logreg']; fig,axes=plt.subplots(1,3,figsize=(14,4),sharey=True)
for ax,(name,sub) in zip(axes,primary.groupby('train_set',sort=False)):
    ax.boxplot([sub.loc[sub.line==line,'tumor_score'] for line in LINE_ORDER],tick_labels=LINE_ORDER,showfliers=False)
    rho=spearmanr(sub.line.map({line:i for i,line in enumerate(LINE_ORDER)}),sub.tumor_score).statistic; ax.set_title(f'{name}: rho={rho:.3f}')
axes[0].set_ylabel('Per-nucleus P(tumor)'); fig.tight_layout()

## Sensitivity to imputed NCO spacing
All imputed rows are removed and the nucleus-level analysis is repeated. Differences from the primary result must be reported.

In [ ]:
measured=akps_raw.loc[~akps_raw.spacing_is_imputed].assign(organoid=lambda x:x.img_name).copy()
measured=stratified_subsample_by_organoid(measured)
scores_measured,summary_measured=run_all(train_sets,measured,modes=('nucleus',))
summary_measured.to_csv(OUT/'notebook_akps_without_imputed_spacing.csv',index=False)
comparison=summary.merge(summary_measured,on=['train_set','model','mode'],suffixes=('_all','_measured_only'))
comparison[['train_set','model','spearman_rho_all','spearman_rho_measured_only']]

## Direct five-stage nucleus models
Ordinal regression and multinomial classification use stratified grouped folds. Predictions remain one per nucleus.

In [ ]:
nuclei=prep_nucleus_level(akps); folds,predictions=cv_evaluate(nuclei)
folds.to_csv(OUT/'notebook_akps_direct_stage_cv.csv',index=False); predictions.to_csv(OUT/'notebook_akps_direct_stage_oof_predictions.csv',index=False); folds

In [ ]:
_,selected=fit_full_and_select(nuclei); pd.Series(selected,name='coefficient').sort_values()